The mechanistic justification for this step is: Establishing the primary dataset and training the PyTorch sequence model to adapt to human behavior.

We will synthesize a sequence of human RPS moves, train an LSTM, print tensor shapes, and save the model.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# 1 = Rock, 2 = Paper, 3 = Scissors
seq_length = 5
data = [1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 1, 2, 1, 2, 3, 1, 2, 3, 2, 3, 1, 2, 3, 1, 2, 3] * 10
X = []
y = []
for i in range(len(data) - seq_length):
    X.append(data[i:i+seq_length])
    y.append(data[i+seq_length] - 1)  # 0-indexed for CrossEntropy

X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
y_tensor = torch.tensor(y, dtype=torch.long)

print(f"X_tensor shape: {X_tensor.shape}")
print(f"y_tensor shape: {y_tensor.shape}")


In [ ]:
class RPSModel(nn.Module):
    def __init__(self):
        super(RPSModel, self).__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=16, batch_first=True)
        self.fc = nn.Linear(16, 3)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

model = RPSModel()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print(f"Model parameters:", sum(p.numel() for p in model.parameters()))


In [ ]:
epochs = 100
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

if torch.cuda.is_available():
    print(f"Memory allocated: {torch.cuda.memory_allocated()}")
else:
    print("CUDA not available, running on CPU.")

torch.save(model.state_dict(), 'rps_lstm_model.pth')
